In [ ]:
# Install if needed
!pip install requests beautifulsoup4 pandas openpyxl

import requests
from bs4 import BeautifulSoup
import pandas as pd
import re
import time

base_url = "https://www.hushpuppies.com.pk"

collection_urls = [
    "https://www.hushpuppies.com.pk/collections/men",
    "https://www.hushpuppies.com.pk/collections/women"
]

headers = {
    "User-Agent": "Mozilla/5.0"
}

all_products = []
product_links = set()

# -----------------------------------
# STEP 1: COLLECT PRODUCT LINKS
# -----------------------------------

for collection_url in collection_urls:

    print(f"\nScanning: {collection_url}")

    response = requests.get(collection_url, headers=headers)

    soup = BeautifulSoup(response.text, "html.parser")

    for link in soup.find_all("a", href=True):

        href = link["href"]

        if "/products/" in href:

            if href.startswith("/"):
                full_link = base_url + href
            else:
                full_link = href

            product_links.add(full_link)

print(f"\nTotal Product Links Found: {len(product_links)}")


# -----------------------------------
# STEP 2: VISIT EACH PRODUCT PAGE
# -----------------------------------

for count, product_url in enumerate(product_links, start=1):

    try:

        print(f"[{count}] Processing")

        response = requests.get(
            product_url,
            headers=headers
        )

        soup = BeautifulSoup(
            response.text,
            "html.parser"
        )

        # -------------------------
        # PRODUCT NAME
        # -------------------------
        title = "N/A"

        h1 = soup.find("h1")

        if h1:
            title = h1.get_text(strip=True)

        # -------------------------
        # PRICE
        # -------------------------
        price = "N/A"

        page_text = soup.get_text(" ", strip=True)

        price_match = re.search(
            r'Rs\.?\s*[\d,]+',
            page_text
        )

        if price_match:
            price = price_match.group()

        # -------------------------
        # CATEGORY
        # -------------------------
        category = "N/A"

        breadcrumbs = soup.find_all("a")

        for item in breadcrumbs:

            txt = item.get_text(strip=True)

            if txt.lower() in [
                "men",
                "women",
                "sale",
                "new arrivals"
            ]:

                category = txt
                break

        # -------------------------
        # DESCRIPTION
        # -------------------------
        description = "N/A"

        meta_desc = soup.find(
            "meta",
            attrs={"name": "description"}
        )

        if meta_desc:
            description = meta_desc.get(
                "content",
                "N/A"
            )

        # -------------------------
        # COLORS
        # -------------------------
        colors = set()

        possible_colors = [
            "black","brown","tan",
            "white","blue","navy",
            "grey","gray","green",
            "red","beige","camel"
        ]

        text_elements = soup.find_all(
            ["label", "span", "option", "div"]
        )

        for element in text_elements:

            txt = element.get_text(
                " ",
                strip=True
            ).lower()

            for color in possible_colors:

                if color == txt:
                    colors.add(color.title())

        # -------------------------
        # SIZES
        # -------------------------
        sizes = set()

        for option in soup.find_all(
            ["option", "label", "span"]
        ):

            txt = option.get_text(
                strip=True
            )

            if txt.isdigit():

                size_num = int(txt)

                if 35 <= size_num <= 50:
                    sizes.add(txt)

        # -------------------------
        # STOCK STATUS
        # -------------------------
        stock_status = "Out of Stock"

        lower_text = page_text.lower()

        if (
            "add to cart" in lower_text
            or "buy now" in lower_text
            or "available" in lower_text
        ):
            stock_status = "In Stock"

        # -------------------------
        # SAVE DATA
        # -------------------------
        all_products.append({

            "Product Name": title,
            "Category": category,
            "Price": price,
            "Colors": ", ".join(colors)
                      if colors else "N/A",
            "Sizes": ", ".join(sizes)
                     if sizes else "N/A",
            "Stock Status": stock_status,
            "Description": description,
            "Product URL": product_url

        })

        print(title)

        time.sleep(0.5)

    except Exception as e:

        print(
            f"Error on {product_url}:",
            e
        )


# -----------------------------------
# STEP 3: EXPORT TO EXCEL
# -----------------------------------

df = pd.DataFrame(all_products)

df.drop_duplicates(inplace=True)

excel_file = "HushPuppies_Dataset.xlsx"

df.to_excel(
    excel_file,
    index=False
)

print("\nDataset Created Successfully")
print("Total Products:", len(df))
print("Saved File:", excel_file)

display(df.head())
from google.colab import files

files.download("HushPuppies_Dataset.xlsx")


Scanning: https://www.hushpuppies.com.pk/collections/men

Scanning: https://www.hushpuppies.com.pk/collections/women

Total Product Links Found: 49
[1] Processing
Cross Wider
[2] Processing
Enzo Levi
[3] Processing
Irving Banker -Waterproof Formal Slip-Ons
[4] Processing
Lama Barrie
[5] Processing
Hash Hype
[6] Processing
Vogue Mali
[7] Processing
Flyknit Nest Piana Sneakers
[8] Processing
Rainmaker- Waterproof Formal Slip-Ons
[9] Processing
Comfy Hoka Sendai Slippers
[10] Processing
Belle LEPF
[11] Processing
Tiana LEPF
[12] Processing
Stocks- Formal Slip-Ons
[13] Processing
Breathable Speed Arrow Sneakers
[14] Processing
Flexible Trim Elise Slippers
[15] Processing
Broox Helix
[16] Processing
Flyknit Moka Piana Sneakers
[17] Processing
Lilly Barie
[18] Processing
Trendy SIA Birken Slippers
[19] Processing
Slink Helix
[20] Processing
Cushioned Peak Kinder Slippers
[21] Processing
Trendy NIX Birken Slippers
[22] Processing
Stamp cobalt
[23] Processing
Cushioned Pump Kinder Slippers
[2

,Product Name,Category,Price,Colors,Sizes,Stock Status,Description,Product URL
0,Cross Wider,Men,"Rs.4,199","Tan, Black","40, 41, 44, 43, 45, 42",In Stock,"A perfect partner, All day long comfort, comfo...",https://www.hushpuppies.com.pk/products/cross-...
1,Enzo Levi,Men,"Rs.5,499",Brown,"40, 41, 44, 43, 45, 42",In Stock,"A perfect partner, All day long comfort, comfo...",https://www.hushpuppies.com.pk/products/enzo-levi
2,Irving Banker -Waterproof Formal Slip-Ons,Men,"Rs.24,999",Black,"40, 41, 44, 43, 45, 42",In Stock,"Incredibly simple and decent, A pair of shoes ...",https://www.hushpuppies.com.pk/products/irving...
3,Lama Barrie,Men,"Rs.2,449",N/A,"39, 40, 36, 41, 37, 38",In Stock,"A perfect partner, All day long comfort, comfo...",https://www.hushpuppies.com.pk/products/lama-b...
4,Hash Hype,Men,"Rs.3,499","Navy, Grey","39, 40, 36, 41, 37, 38",In Stock,"A perfect partner, All day long comfort, comfo...",https://www.hushpuppies.com.pk/products/hash-hype


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>